<a href="https://colab.research.google.com/github/sanikaa12-art/hinglish-sarcasm-muril/blob/main/hinglish_sarcasm_muril.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -q https://github.com/sahilswami96/SarcasmDetection_CodeMixed
import pandas as pd
from sklearn.model_selection import train_test_split
D = 'SarcasmDetection_CodeMixed/Dataset/'

def blocks(path):
    raw = open(path, encoding='utf-8').read().strip().split('\n\n')
    return [b.strip().split('\n') for b in raw if b.strip()]
texts = {b[0].strip(): ' '.join(b[1:]).strip() for b in blocks(D + 'Sarcasm_tweets.txt')}
lines = [l.strip() for l in open(D + 'Sarcasm_tweet_truth.txt', encoding='utf-8') if l.strip()]
labels = {lines[i]: lines[i+1] for i in range(0, len(lines), 2)}

df = pd.DataFrame([(k, texts[k], 1 if labels[k] == 'YES' else 0) for k in texts], columns=['id','text','label'])
train, temp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp.label, random_state=42)
print(len(train), len(val), len(test)); print(train.label.mean(), test.label.mean())
df.sample(5, random_state=1)

3675 787 788
0.09605442176870749 0.09644670050761421


,id,text,label
4,866264330748219392,#RememberingRajiv aaj agar musalman auraten tr...,0
3248,906106087945928704,Naamwar Bollywood adakaara Sunny Leone ka kehn...,0
440,895662711488434176,"From ""Gadar - Ek prem katha"" to ""Toilet - ek p...",0
1413,869608307434115072,"Abe suar ki aulad,TRIPLE TALAQ nd HALALA ki sa...",0
4351,899994002161795072,Yaar kabhi toh minority appeasment se upar uth...,0


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2)
Xtr, Xte = vec.fit_transform(train.text), vec.transform(test.text)
clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, train.label)
pred = clf.predict(Xte)

print('Baseline macro-F1:', round(f1_score(test.label, pred, average='macro'), 4))
print(classification_report(test.label, pred, digits=3, target_names=['not sarcastic','sarcastic']))

Baseline macro-F1: 0.8954
               precision    recall  f1-score   support

not sarcastic      0.999     0.954     0.976       712
    sarcastic      0.694     0.987     0.815        76

     accuracy                          0.957       788
    macro avg      0.846     0.970     0.895       788
 weighted avg      0.969     0.957     0.960       788



In [3]:
import re
LEAK = re.compile(r'#?\b(sarcas\w*|irony|ironic\w*)\b', re.I)
for d in (train, val, test):
    d['clean'] = d.text.apply(lambda t: LEAK.sub(' ', t).strip())

print('Leak words left:', sum(test.clean.str.contains(LEAK)))

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2)
Xtr, Xte = vec.fit_transform(train.clean), vec.transform(test.clean)
clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, train.label)
pred = clf.predict(Xte)
print('Clean baseline macro-F1:', round(f1_score(test.label, pred, average='macro'), 4))
print(classification_report(test.label, pred, digits=3, target_names=['not sarcastic','sarcastic']))

/tmp/ipykernel_485/4019678907.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print('Leak words left:', sum(test.clean.str.contains(LEAK)))


Leak words left: 0
Clean baseline macro-F1: 0.8882
               precision    recall  f1-score   support

not sarcastic      0.996     0.954     0.974       712
    sarcastic      0.689     0.961     0.802        76

     accuracy                          0.954       788
    macro avg      0.842     0.957     0.888       788
 weighted avg      0.966     0.954     0.958       788



In [4]:
for d in (train, val, test):
    d['clean'] = d.clean.str.replace(r'\s+', ' ', regex=True).str.strip()


In [6]:
!pip -q install transformers datasets accelerate
import torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report

MODEL = 'google/muril-base-cased'
tok = AutoTokenizer.from_pretrained(MODEL)
enc = lambda b: tok(b['clean'], truncation=True, max_length=64, padding='max_length')
ds = {n: Dataset.from_pandas(d[['clean','label']].reset_index(drop=True)).map(enc, batched=True)
      for n, d in [('train',train),('val',val),('test',test)]}

w = torch.tensor([1.0, (train.label==0).sum()/(train.label==1).sum()], dtype=torch.float)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'sarcasm_f1': f1_score(p.label_ids, pred)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments('muril-out', eval_strategy='epoch', save_strategy='epoch', learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=4, load_best_model_at_end=True,
    metric_for_best_model='macro_f1', seed=42, report_to='none')
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'], compute_metrics=metrics)
trainer.train()

p = trainer.predict(ds['test'])
print('MuRIL macro-F1:', round(p.metrics['test_macro_f1'], 4))
print(classification_report(test.label, p.predictions.argmax(-1), digits=3, target_names=['not sarcastic','sarcastic']))

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/3675 [00:00<?, ? examples/s]

Map:   0%|          | 0/787 [00:00<?, ? examples/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MuRIL macro-F1: 0.8875
               precision    recall  f1-score   support

not sarcastic      0.991     0.959     0.975       712
    sarcastic      0.707     0.921     0.800        76

     accuracy                          0.956       788
    macro avg      0.849     0.940     0.888       788
 weighted avg      0.964     0.956     0.958       788



In [7]:
!git clone -q https://github.com/LCS2-IIITD/MSH-COMICS
import pickle, numpy.core.multiarray as ncm
# Safe loader: only allows pandas/numpy objects (pickles can otherwise run code)
OK = {('builtins','slice'),('numpy','dtype'),('numpy','ndarray'),('numpy.core.multiarray','_reconstruct'),
      ('pandas.core.frame','DataFrame'),('pandas.core.indexes.base','Index'),('pandas.core.indexes.base','_new_Index'),
      ('pandas.core.indexes.range','RangeIndex'),('pandas.core.internals.managers','BlockManager')}
class SafeLoader(pickle.Unpickler):
    def find_class(self, m, n):
        if (m, n) not in OK: raise pickle.UnpicklingError(f'blocked {m}.{n}')
        return getattr(ncm, n) if m == 'numpy.core.multiarray' else super().find_class(m, n)

masac = SafeLoader(open('MSH-COMICS/Data/Pickles/Dataset.p','rb')).load()
masac = masac[['text','Sarcasm']].rename(columns={'Sarcasm':'label'}).dropna()
masac['clean'] = masac.text.str.replace(r'\s+', ' ', regex=True).str.strip()
ood = masac.sample(2000, random_state=42).reset_index(drop=True)
print(len(ood), ood.label.mean())

# Baseline trained on tweets -> tested on TV dialogue
bp = clf.predict(vec.transform(ood.clean))
print('Baseline cross-domain macro-F1:', round(f1_score(ood.label, bp, average='macro'), 4))

# MuRIL trained on tweets -> tested on TV dialogue
ood_ds = Dataset.from_pandas(ood[['clean','label']]).map(enc, batched=True)
mp = trainer.predict(ood_ds).predictions.argmax(-1)
print('MuRIL cross-domain macro-F1:', round(f1_score(ood.label, mp, average='macro'), 4))
print(classification_report(ood.label, mp, digits=3, target_names=['not sarcastic','sarcastic']))

2000 0.205
Baseline cross-domain macro-F1: 0.215


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

MuRIL cross-domain macro-F1: 0.1825
               precision    recall  f1-score   support

not sarcastic      0.697     0.014     0.028      1590
    sarcastic      0.203     0.976     0.337       410

     accuracy                          0.211      2000
    macro avg      0.450     0.495     0.182      2000
 weighted avg      0.596     0.211     0.092      2000



In [8]:
from sklearn.model_selection import GroupShuffleSplit
m = SafeLoader(open('MSH-COMICS/Data/Pickles/Dataset.p','rb')).load()
m = m[['text','Sarcasm','Episode_label']].rename(columns={'Sarcasm':'label'}).dropna()
m['clean'] = m.text.str.replace(r'\s+', ' ', regex=True).str.strip()
m['ep'] = m.Episode_label.astype(str).str.split('-').str[0]

g = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42); a, b = next(g.split(m, groups=m.ep))
m_tr, tmp = m.iloc[a], m.iloc[b]
g2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42); c, d = next(g2.split(tmp, groups=tmp.ep))
m_val, m_te = tmp.iloc[c], tmp.iloc[d]
print(len(m_tr), len(m_val), len(m_te), round(m_te.label.mean(), 3))

vec2 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2)
clf2 = LogisticRegression(max_iter=2000, class_weight='balanced').fit(vec2.fit_transform(m_tr.clean), m_tr.label)
print('Baseline (MaSaC-trained) on MaSaC test:', round(f1_score(m_te.label, clf2.predict(vec2.transform(m_te.clean)), average='macro'), 4))

10816 2497 2249 0.194
Baseline (MaSaC-trained) on MaSaC test: 0.5334


In [9]:
tweet_trainer = trainer   # keep the tweet-trained model
enc = lambda b: tok(b['clean'], truncation=True, max_length=128, padding='max_length')
mk = lambda d: Dataset.from_pandas(d[['clean','label']].reset_index(drop=True)).map(enc, batched=True)
m_ds = {'train': mk(m_tr), 'val': mk(m_val), 'test': mk(m_te)}
w = torch.tensor([1.0, (m_tr.label==0).sum()/(m_tr.label==1).sum()], dtype=torch.float)

model2 = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args2 = TrainingArguments('muril-masac', eval_strategy='epoch', save_strategy='epoch', learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=3, load_best_model_at_end=True,
    metric_for_best_model='macro_f1', seed=42, report_to='none')
masac_trainer = WeightedTrainer(model=model2, args=args2, train_dataset=m_ds['train'], eval_dataset=m_ds['val'], compute_metrics=metrics)
masac_trainer.train()

def score(tr, ds, y): p = tr.predict(ds).predictions.argmax(-1); return round(f1_score(y, p, average='macro'), 4), p
s1, p1 = score(masac_trainer, m_ds['test'], m_te.label)
s2, _  = score(masac_trainer, mk(test), test.label)
s3, _  = score(tweet_trainer, m_ds['test'], m_te.label)
print('MuRIL MaSaC→MaSaC:', s1, '| MaSaC→Tweets:', s2, '| Tweets→MaSaC (same test):', s3)
print(classification_report(m_te.label, p1, digits=3, target_names=['not sarcastic','sarcastic']))

Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2249 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Epoch,Training Loss,Validation Loss,Macro F1,Sarcasm F1
1,No log,0.673760,0.568894,0.337370
2,0.682614,0.670325,0.575535,0.350128
3,0.666661,0.667760,0.561476,0.344937


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

MuRIL MaSaC→MaSaC: 0.5432 | MaSaC→Tweets: 0.4854 | Tweets→MaSaC (same test): 0.1785
               precision    recall  f1-score   support

not sarcastic      0.827     0.770     0.797      1812
    sarcastic      0.257     0.330     0.289       437

     accuracy                          0.685      2249
    macro avg      0.542     0.550     0.543      2249
 weighted avg      0.716     0.685     0.699      2249



In [11]:
m = m.reset_index(drop=True)
ctx = []
for i in range(len(m)):
    prev = [m.clean.iloc[j] for j in range(max(0, i-2), i) if m.Episode_label.iloc[j] == m.Episode_label.iloc[i]]
    ctx.append(' | '.join(prev))
m['ctx'] = ctx
m_tr, tmp = m.iloc[a], m.iloc[b]
m_val, m_te = tmp.iloc[c], tmp.iloc[d]
print(m_te[['ctx','clean','label']].head(3))

enc_ctx = lambda b: tok(b['ctx'], b['clean'], truncation='longest_first', max_length=160, padding='max_length')
mkc = lambda d: Dataset.from_pandas(d[['ctx','clean','label']].reset_index(drop=True)).map(enc_ctx, batched=True)
c_ds = {'train': mkc(m_tr), 'val': mkc(m_val), 'test': mkc(m_te)}

model3 = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args3 = TrainingArguments('muril-ctx', eval_strategy='epoch', save_strategy='epoch', learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=3, load_best_model_at_end=True,
    metric_for_best_model='macro_f1', seed=42, report_to='none')
ctx_trainer = WeightedTrainer(model=model3, args=args3, train_dataset=c_ds['train'], eval_dataset=c_ds['val'], compute_metrics=metrics)
ctx_trainer.train()
s4, p4 = score(ctx_trainer, c_ds['test'], m_te.label)
print('MuRIL + context, MaSaC→MaSaC:', s4, '(no context was 0.5432)')
print(classification_report(m_te.label, p4, digits=3, target_names=['not sarcastic','sarcastic']))

                                                   ctx  \
133                                                      
134           नहीं. नहीं, नहीं. मैने पीना बंद कर दिया.   
135  नहीं. नहीं, नहीं. मैने पीना बंद कर दिया. | क्यों?   

                                            clean  label  
133      नहीं. नहीं, नहीं. मैने पीना बंद कर दिया.      0  
134                                        क्यों?      1  
135  क्योंकि Maya ने मुझसे बात करना बंद कर दी है.      0  


Map:   0%|          | 0/10816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2249 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Epoch,Training Loss,Validation Loss,Macro F1,Sarcasm F1
1,No log,0.668209,0.569339,0.330638
2,0.678257,0.658667,0.561748,0.351310
3,0.658368,0.662751,0.552637,0.362369


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MuRIL + context, MaSaC→MaSaC: 0.5434 (no context was 0.5432)
               precision    recall  f1-score   support

not sarcastic      0.825     0.786     0.805      1812
    sarcastic      0.259     0.309     0.282       437

     accuracy                          0.694      2249
    macro avg      0.542     0.548     0.543      2249
 weighted avg      0.715     0.694     0.704      2249

